# Credit Card Approval Prediction - Model Comparison

This notebook compares the performances of **Logistic Regression**, **Decision Tree**, and **Random Forest** and selects the best model for web integration.

## 1. Import Core Libraries and Load Preprocessed Data

In [ ]:
import os
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, roc_curve

# Load processed dataset
df = pd.read_csv(os.path.join('..', '06_Data_Preprocessing', 'processed_dataset.csv'))
X = df.drop(columns=['y'])
y = df['y']

## 2. Train and Evaluate Models

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

models = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'Decision Tree': DecisionTreeClassifier(random_state=42, max_depth=6),
    'Random Forest': RandomForestClassifier(random_state=42, n_estimators=100, max_depth=10)
}

results = {}

for name, clf in models.items():
    start_train = time.time()
    clf.fit(X_train, y_train)
    train_time = time.time() - start_train
    
    start_pred = time.time()
    y_pred = clf.predict(X_test)
    pred_time = time.time() - start_pred
    
    y_prob = clf.predict_proba(X_test)[:, 1]
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    auc = roc_auc_score(y_test, y_prob)
    
    results[name] = {
        'model': clf,
        'accuracy': acc,
        'precision': prec,
        'recall': rec,
        'f1': f1,
        'auc': auc,
        'train_time': train_time,
        'pred_time': pred_time,
        'y_prob': y_prob
    }

## 3. Compare Models

In [ ]:
comparison_table = []
for name, res in results.items():
    comparison_table.append({
        'Model Name': name,
        'Accuracy': f"{res['accuracy']:.4f}",
        'Precision': f"{res['precision']:.4f}",
        'Recall': f"{res['recall']:.4f}",
        'F1 Score': f"{res['f1']:.4f}",
        'Training Time (s)': f"{res['train_time']:.4f}",
        'Prediction Time (s)': f"{res['pred_time']:.4f}"
    })

comparison_df = pd.DataFrame(comparison_table)
print(comparison_df.to_string(index=False))

## 4. Visualizations

In [ ]:
# Accuracy Comparison Plot
plt.figure(figsize=(8, 5))
sns.barplot(x='Model Name', y='Accuracy', data=comparison_df.assign(Accuracy=comparison_df['Accuracy'].astype(float)), palette='Blues_d')
plt.title('Model Accuracy Comparison', fontsize=14, fontweight='bold', color='#1E3A8A')
plt.ylim(0.5, 1.02)
plt.ylabel('Accuracy')
plt.savefig('outputs/accuracy_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

# ROC Curve Comparison Plot
plt.figure(figsize=(8, 6))
for name, res in results.items():
    fpr, tpr, _ = roc_curve(y_test, res['y_prob'])
    plt.plot(fpr, tpr, label=f"{name} (AUC = {res['auc']:.4f})")
plt.plot([0, 1], [0, 1], 'k--', alpha=0.5)
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves Comparison', fontsize=14, fontweight='bold', color='#1E3A8A')
plt.legend(loc='lower right')
plt.savefig('outputs/roc_curve.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Save the Best Performing Model

In [ ]:
# Choose best model
best_model_name = max(results.keys(), key=lambda name: (results[name]['accuracy'], results[name]['f1']))
print(f"Best Performing Model: {best_model_name}")

best_model = results[best_model_name]['model']

# Save model
os.makedirs('../08_Web_Application/model', exist_ok=True)
joblib.dump(best_model, '../08_Web_Application/model/model.pkl')
joblib.dump(best_model, 'best_model.pkl')
print("Best model successfully saved.")